# PHYS 338 — Physique Statistique — Homework 1
## Information, Probabilités, Entropie

**Nom :** ...
**Date :** ...


### Dépendances
- Python ≥ 3.9
- `numpy`, `matplotlib`, `scipy`
- modules standard : `os`, `zlib`, `bz2`, `lzma`, `collections`

In [1]:
import os
import zlib, bz2, lzma
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=338)  # graine fixée pour la reproductibilité

---
# Exercice 1 — Entropie et compression des données

## Part I — Surprise, incertitude et information

Variable aléatoire $X$ à valeurs dans un alphabet $\chi$ de $L$ lettres, $P(X = x_i) = p_i$.

$\chi = \{A, B, C, D\}$ avec $p_A = 1/2,\ p_B = 1/4,\ p_C = 1/8,\ p_D = 1/8$.

### Q1 — Incertitude $H[X]$
Surprise d'un tirage : $\log_2 (1/p_i)$. L'incertitude $H[X]$ est la surprise moyenne.
Comment s'écrit l'incertitude en général ? Que vaut-elle dans notre exemple ?

**Réponse :**

**Cas général**:  
La surprise associée au tirage $X = x_i$ est $\log_2 \frac{1}{p_i}$. L'incertitude $H[X]$ est la moyenne de la surprise étant pondérée par sa probabilité $p_i$ :

$$
H[X] = \sum_{i=1}^{L} p_i \log_2 \frac{1}{p_i} = -\sum_{i=1}^{L} p_i \log_2 p_i
$$


**Dans notre exemple**:


$$
H[X] = \frac{1}{2} + \frac{1}{2} + \frac{3}{8} + \frac{3}{8} = \frac{7}{4}.
$$

### Q2 — Bornes de $H[X]$
Montrer que $0 \le H[X] \le \log_2 L$ ; que $0$ est atteint ssi une seule lettre a une probabilité non nulle ;
que $\log_2 L$ est atteint ssi toutes les lettres sont équiprobables.

**Réponse :**


**Borne inférieure : $H[X] \ge 0$.** Pour tout $i$, $0 \le p_i \le 1$, donc $\log_2 \frac{1}{p_i} \ge 0$ et chaque terme vérifie $p_i \log_2 \frac{1}{p_i} \ge 0$. Une somme de termes positifs est positive :

$$
H[X] = \sum_{i=1}^{L} p_i \log_2 \frac{1}{p_i} \ge 0.
$$

**Cas d'égalité $H[X] = 0$.** Une somme de termes positifs est nulle si et seulement si chaque terme est nul. Or $p_i \log_2 \frac{1}{p_i} = 0$ si et seulement si $p_i = 0$ ou $p_i = 1$. Comme $\sum_i p_i = 1$, cela impose qu'une seule lettre ait une probabilité non nulle. Réciproquement, si une seule lettre a $p_j = 1$ et toutes les autres $p_i = 0$, tous les termes sont nuls et $H[X] = 0$.

**Borne supérieure : $H[X] \le \log_2 L$.** On utilise l'inégalité $\ln x \le x - 1$, valable pour tout $x > 0$, avec égalité si et seulement si $x = 1$. En notant $S = \{i : p_i > 0\}$ l'ensemble des lettres de probabilité non nulle :

$$
H[X] - \log_2 L = \sum_{i \in S} p_i \log_2 \frac{1}{L\,p_i} = \frac{1}{\ln 2} \sum_{i \in S} p_i \ln \frac{1}{L\,p_i}
\le \frac{1}{\ln 2} \sum_{i \in S} p_i \left( \frac{1}{L\,p_i} - 1 \right).
$$

Or

$$
\sum_{i \in S} p_i \left( \frac{1}{L\,p_i} - 1 \right) = \frac{|S|}{L} - \sum_{i \in S} p_i = \frac{|S|}{L} - 1 \le 0,
$$

car $|S| \le L$. D'où $H[X] \le \log_2 L$.

**Cas d'égalité $H[X] = \log_2 L$.** Les deux inégalités doivent être des égalités :
- $\ln x = x - 1$ impose $\frac{1}{L\,p_i} = 1$, soit $p_i = \frac{1}{L}$ pour tout $i \in S$ ;
- $\frac{|S|}{L} = 1$ impose $|S| = L$, c'est-à-dire que toutes les lettres ont une probabilité non nulle.

Donc $p_i = \frac{1}{L}$ pour tout $i$ : **toutes les lettres sont équiprobables**. Réciproquement, dans ce cas $H[X] = \sum_{i=1}^{L} \frac{1}{L} \log_2 L = \log_2 L$.

**Interprétation.** L'incertitude est nulle quand le résultat est certain, et maximale quand on n'a aucune raison de préférer une lettre à une autre. Dans notre exemple, $H[X] = 1{,}75 < \log_2 4 = 2$ bits, car la distribution n'est pas uniforme.

### Q3 — Information manquante
L'information manquante sur la valeur secrète $x_i$ est $\log_2(1/p_i)$.
Quelle est sa valeur moyenne en général ? Que vaut-elle dans notre exemple ?

**Réponse :**

**Cas général.** Si la valeur mesurée est $x_i$, l'information qu'il vous manque est $\log_2 \frac{1}{p_i}$. Cette valeur $x_i$ apparaît avec probabilité $p_i$, donc l'information manquante moyenne est l'espérance :

$$
\langle I \rangle = \sum_{i=1}^{L} p_i \log_2 \frac{1}{p_i} = -\sum_{i=1}^{L} p_i \log_2 p_i = H[X]
$$

C'est **exactement la même expression que l'incertitude** $H[X]$ de la Q1. Les deux points de vue décrivent la même quantité, vue avant ou après le tirage :
- *avant* le tirage, $H[X]$ mesure votre incertitude moyenne, c'est-à-dire la surprise que vous aurez en moyenne ;
- *après* le tirage, $H[X]$ mesure l'information moyenne qu'il vous faut recevoir pour connaître le résultat.

L'information apportée en révélant $x_i$ est donc précisément ce qui lève votre incertitude : l'entropie de Shannon s'interprète indifféremment comme une mesure d'**incertitude** ou d'**information manquante**.

**Dans notre exemple :**

$$
\langle I \rangle = \frac{1}{2}\cdot 1 + \frac{1}{4}\cdot 2 + \frac{1}{8}\cdot 3 + \frac{1}{8}\cdot 3 = \frac{7}{4} = 1{,}75 \text{ bits}.
$$

À comparer avec $\log_2 4 = 2$ bits pour quatre lettres équiprobables : connaître les probabilités vous donne déjà une partie de l'information, donc il vous en manque moins en moyenne.

## Part II — Entropie de Shannon et questions OUI/NON

### Q1 — Stratégie de questions pour {A, B, C, D}
Combien de questions OUI/NON faut-il poser en moyenne pour trouver la lettre tirée ?
Comparer avec l'entropie de l'alphabet.

**Réponse :**


**Stratégie.** Pour trouver la lettre le plus vite possible en moyenne, chaque question doit apporter le maximum d'information. On découpe donc à chaque étape les lettres restantes en deux groupes de probabilités égales, et on demande si la lettre appartient au premier groupe. Chaque réponse est alors équiprobable et apporte exactement 1 bit d'information.

Avec $p_A = 1/2$, $p_B = 1/4$, $p_C = p_D = 1/8$ :

1. « La lettre est-elle dans $\{A\}$ ? » : $\{A\}$ contre $\{B, C, D\}$, soit $1/2$ contre $1/2$. Si OUI, on a trouvé en 1 question.
2. Sinon, « La lettre est-elle dans $\{B\}$ ? » : $\{B\}$ contre $\{C, D\}$, soit $1/4$ contre $1/4$. Si OUI, on a trouvé en 2 questions.
3. Sinon, « La lettre est-elle dans $\{C\}$ ? » : la réponse donne C ou D en 3 questions.


**Nombre moyen de questions.**

$$
\langle n \rangle = \sum_{i} p_i \, n_i = \frac{1}{2}\cdot 1 + \frac{1}{4}\cdot 2 + \frac{1}{8}\cdot 3 + \frac{1}{8}\cdot 3 = \frac{7}{4} = H[X]
$$


On obtient donc $\langle n \rangle = H[X]$ : le nombre moyen de questions est exactement égal à l'entropie. On remarque aussi que chaque lettre est trouvée en $n_i = \log_2 1/p_i$ questions, c'est-à-dire que le nombre de questions pour une lettre est égal à sa surprise.



### Q2 — Alphabet de $2^b$ lettres équiprobables
Combien de questions faut-il poser en moyenne ? Comparer avec l'entropie du système.

**Réponse :**

**Stratégie.** Toutes les lettres ont la même probabilité $p_i = 1/2^b$. Pour couper en deux groupes équiprobables, il suffit de couper l'ensemble des lettres restantes **en deux moitiés de même taille** (recherche dichotomique) :

- avant la 1re question : $2^b$ lettres possibles ;
- après la 1re question : $2^{b-1}$ lettres ;
- après la $k$-ième question : $2^{b-k}$ lettres.

La lettre est identifiée quand il ne reste plus qu'une seule lettre, soit $2^{b-k} = 1$, donc après $k = b$ questions.

**Nombre moyen de questions.** Chaque lettre demande exactement $b$ questions, quelle que soit la lettre tirée :

$$
\langle n \rangle = \sum_{i=1}^{2^b} \frac{1}{2^b} \cdot b = b.
$$

**Comparaison avec l'entropie.** Pour une distribution uniforme sur $L = 2^b$ lettres :

$$
H[X] = \sum_{i=1}^{2^b} \frac{1}{2^b} \log_2 2^b = \log_2 2^b = b .
$$

On retrouve $\langle n \rangle = H[X] = b$. C'est aussi la valeur maximale $\log_2 L$ de l'entropie (Partie I, Q2) : l'alphabet uniforme est le plus difficile à deviner.

*Remarque :* en notant OUI = 1 et NON = 0, la suite des $b$ réponses est simplement l'écriture binaire de l'indice de la lettre sur $b$ bits. Sans information sur les probabilités, on ne peut pas faire mieux qu'un code de longueur fixe.

### Q3 — Théorème de Shannon (1948)
Alphabet général de $A$ lettres de probabilités $p_i$. Comment généraliser la stratégie ?
Combien de questions en moyenne ? (On suppose qu'on peut toujours couper en deux groupes de probabilité 1/2.)

**Réponse :**

**Généralisation de la stratégie.** On considère $A$ lettres $x_i$ de probabilités $p_i$. À chaque étape, on partage les lettres restantes en **deux groupes de même probabilité** (ce qui est possible par hypothèse), et on demande si la lettre tirée appartient au premier groupe. Chaque réponse OUI/NON est équiprobable et apporte donc exactement 1 bit d'information.

**Nombre de questions pour une lettre donnée.** Au départ, l'ensemble des lettres a une probabilité totale de $1$. Chaque question divise par deux la probabilité du groupe dans lequel se trouve la lettre :

$$
\text{après } k \text{ questions, le groupe restant a une probabilité } 2^{-k}.
$$

La lettre $x_i$ est identifiée lorsque le groupe restant se réduit à $\{x_i\}$, c'est-à-dire lorsque sa probabilité vaut $p_i$ :

$$
2^{-n_i} = p_i \quad \Longrightarrow \quad n_i = \log_2 \frac{1}{p_i}.
$$

Le nombre de questions nécessaires pour trouver $x_i$ est donc exactement sa **surprise**.

**Nombre moyen de questions.**

$$
\langle n \rangle = \sum_{i=1}^{A} p_i \, n_i = \sum_{i=1}^{A} p_i \log_2 \frac{1}{p_i} = H[X].
$$

**On ne peut pas faire mieux.** Une stratégie de questions correspond à un code binaire préfixe de longueurs $n_i$, qui vérifie l'inégalité de Kraft $\sum_i 2^{-n_i} \le 1$. En utilisant $\ln x \le x - 1$ :

$$
H[X] - \langle n \rangle = \sum_i p_i \log_2 \frac{2^{-n_i}}{p_i} \le \frac{1}{\ln 2} \sum_i p_i \left( \frac{2^{-n_i}}{p_i} - 1 \right) = \frac{1}{\ln 2}\left( \sum_i 2^{-n_i} - 1 \right) \le 0.
$$

Donc $\langle n \rangle \ge H[X]$ pour **toute** stratégie : l'entropie est le nombre minimal de questions OUI/NON nécessaires en moyenne, et la stratégie ci-dessus atteint cette borne. Intuitivement, une question OUI/NON apporte au plus 1 bit d'information, et ce maximum n'est atteint que si les deux réponses sont équiprobables.

**Cas général.** L'hypothèse d'un partage toujours exact en deux moitiés revient à supposer que toutes les $p_i$ sont des puissances de $1/2$. Sinon, $\log_2 1/p_i$ n'est pas entier, et on prend $n_i = \lceil \log_2 1/p_i \rceil$, ce qui donne :

$$
H[X] \le \langle n \rangle < H[X] + 1.
$$

En codant des blocs de $k$ lettres à la fois, le surcoût de moins d'un bit est réparti sur $k$ lettres, et le nombre moyen de questions **par lettre** tend vers $H[X]$ quand $k \to \infty$. C'est le **théorème du codage de source de Shannon (1948)** : l'entropie est la limite ultime de la compression sans perte.

## Part III — Compression de données
Fichier de $N$ valeurs dans un alphabet de $L$ lettres : taille $N \log_2 L$ bits.

### Q1 — Compression par questions OUI/NON
On remplace chaque lettre par la suite de ses réponses (OUI = 1, NON = 0).
Taille du nouveau fichier ? Facteur de compression (taille avant / taille après) ?

**Réponse :**

**Taille du fichier compressé.** D'après la Partie II, avec la stratégie optimale, la lettre $x_i$ est identifiée en $n_i = \log_2 \frac{1}{p_i}$ questions. On remplace donc chaque occurrence de $x_i$ par une suite de $n_i$ bits (OUI = 1, NON = 0). Pour l'alphabet de l'exemple : A = `1`, B = `01`, C = `001`, D = `000`.

Le code est **préfixe** (aucun mot de code n'est le début d'un autre) : on peut relire le fichier bit par bit et savoir sans ambiguïté où finit chaque lettre, sans séparateur. La compression est donc **sans perte**.

Parmi les $N$ lettres du fichier, la lettre $x_i$ apparaît en moyenne $N p_i$ fois. La taille moyenne du fichier compressé est donc

$$
T_{\text{après}} = \sum_i (N p_i)\, n_i = N \sum_i p_i \log_2 \frac{1}{p_i} = N\, H[X] \ \text{bits}.
$$

L'entropie est donc le nombre de bits par lettre du fichier compressé. Les fluctuations autour de cette moyenne sont en $O(\sqrt{N})$, donc négligeables en relatif pour $N$ grand.

**Facteur de compression.**

$$
\tau = \frac{T_{\text{avant}}}{T_{\text{après}}} = \frac{N \log_2 L}{N\, H[X]} = \frac{\log_2 L}{H[X]} \ \ge 1,
$$

car $H[X] \le \log_2 L$ (Partie I, Q2). On a égalité ($\tau = 1$, aucun gain) si et seulement si les lettres sont équiprobables. Plus la distribution est inégale, plus l'entropie est faible et plus on compresse.

**Exemple {A, B, C, D}.** $\tau = \dfrac{\log_2 4}{7/4} = \dfrac{2}{1{,}75} = \dfrac{8}{7} \approx 1{,}14$.

**Cas des fichiers ASCII.** En pratique, chaque caractère d'un fichier texte occupe 8 bits, même si l'alphabet réel contient $L < 256$ lettres. La taille de départ est donc $8N$ bits, et le facteur de compression idéal est $\tau = 8 / H[X]$. Il combine deux gains : on supprime les bits inutilisés ($8 \to \log_2 L$), puis on exploite les probabilités inégales ($\log_2 L \to H$). Pour les fichiers de la Q2 :

| Fichier | Alphabet | $H$ (bits/caractère) | $\tau = 8/H$ |
|:---|:---|:---:|:---:|
| `uniform.txt` | 256 caractères équiprobables | 8 | 1 |
| `half.txt` | 128 caractères équiprobables | 7 | $8/7 \approx 1{,}14$ |
| `abcd.txt` | ABCD équiprobables | 2 | 4 |
| `abcd2.txt` | ABCD, $p = (1/2, 1/4, 1/8, 1/8)$ | 1,75 | $8/1{,}75 \approx 4{,}57$ |

### \* Q2 — Comparaison avec un compresseur usuel
Fichiers de $10^4$ caractères ASCII (8 bits) :
1. `uniform.txt` : parmi les 256 caractères
2. `half.txt` : parmi 128 caractères
3. `abcd.txt` : parmi ABCD (équiprobables)
4. `abcd2.txt` : parmi ABCD avec les probabilités de la partie II

> Placer les fichiers dans le même dossier que ce notebook (ou modifier `DATA_DIR`).

In [ ]:
DATA_DIR = "."
fichiers = ["uniform.txt", "half.txt", "abcd.txt", "abcd2.txt"]
N_CARACTERES = 10**4

def lire_octets(nom):
    # Lit le fichier en binaire (bytes)
    chemin = os.path.join(DATA_DIR, nom)
    with open(chemin, "rb") as f:
        return f.read()

def generer_fichier(nom, N=N_CARACTERES):
    # Génère une suite de N octets selon l'alphabet décrit dans l'énoncé
    # (utilisé seulement si le fichier fourni est absent)
    if nom == "uniform.txt":
        return rng.integers(0, 256, N, dtype=np.uint8).tobytes()
    if nom == "half.txt":
        return rng.integers(0, 128, N, dtype=np.uint8).tobytes()
    if nom == "abcd.txt":
        return rng.choice(np.frombuffer(b"ABCD", dtype=np.uint8), N).tobytes()
    if nom == "abcd2.txt":
        return rng.choice(np.frombuffer(b"ABCD", dtype=np.uint8), N,
                          p=[1/2, 1/4, 1/8, 1/8]).tobytes()
    raise ValueError(nom)

donnees = {}
for nom in fichiers:
    if os.path.exists(os.path.join(DATA_DIR, nom)):
        donnees[nom] = lire_octets(nom)
        source = "fichier fourni"
    else:
        donnees[nom] = generer_fichier(nom)
        source = "ATTENTION : fichier absent, données simulées"
    print(f"{nom:12s} : {len(donnees[nom])} octets  ({source})")

In [ ]:
def entropie_empirique(octets):
    # Entropie (bits/caractère) calculée à partir des fréquences observées dans le fichier
    comptes = np.array(list(Counter(octets).values()), dtype=float)
    f = comptes / comptes.sum()
    return -np.sum(f * np.log2(f))

compresseurs = {
    "zlib (zip)": lambda d: zlib.compress(d, 9),
    "bz2":        lambda d: bz2.compress(d, 9),
    "lzma (xz)":  lambda d: lzma.compress(d, preset=9),
}

# Entropie théorique de chaque alphabet (cf. Q1)
H_theorique = {"uniform.txt": 8.0, "half.txt": 7.0, "abcd.txt": 2.0, "abcd2.txt": 1.75}

resultats = {}
for nom, d in donnees.items():
    taille_avant = 8 * len(d)                       # bits
    H_emp = entropie_empirique(d)
    res = {
        "L observé": len(set(d)),
        "H théorique": H_theorique[nom],
        "H empirique": H_emp,
        "tau prédit (8/H)": 8 / H_theorique[nom],
    }
    for nom_c, comp in compresseurs.items():
        taille_apres = 8 * len(comp(d))
        res[nom_c] = taille_avant / taille_apres    # facteur de compression mesuré
    resultats[nom] = res

colonnes = list(next(iter(resultats.values())).keys())
print(f"{'fichier':12s}" + "".join(f"{c:>18s}" for c in colonnes))
for nom, res in resultats.items():
    print(f"{nom:12s}" + "".join(f"{v:18.3f}" if isinstance(v, float) else f"{v:18d}"
                                 for v in res.values()))

In [ ]:
# Codage OUI/NON (Partie II) appliqué réellement à abcd2.txt : taille obtenue
code_ouinon = {ord("A"): "1", ord("B"): "01", ord("C"): "001", ord("D"): "000"}
d = donnees["abcd2.txt"]
flux = "".join(code_ouinon[c] for c in d)
print(f"abcd2.txt codé OUI/NON : {len(flux)} bits = {len(flux)/len(d):.4f} bits/caractère")
print(f"facteur de compression obtenu : {8*len(d)/len(flux):.3f}  (prédit : {8/1.75:.3f})")

# Graphique : facteurs mesurés vs prédits
noms = list(resultats.keys())
x = np.arange(len(noms))
largeur = 0.2
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 1.5*largeur, [resultats[n]["tau prédit (8/H)"] for n in noms], largeur,
       label="prédit 8/H", color="k")
for k, nom_c in enumerate(compresseurs):
    ax.bar(x + (k - 0.5)*largeur, [resultats[n][nom_c] for n in noms], largeur, label=nom_c)
ax.axhline(1, color="gray", lw=0.8, ls="--")
ax.set_xticks(x)
ax.set_xticklabels(noms)
ax.set_ylabel("facteur de compression")
ax.set_title("Compresseurs usuels vs limite de Shannon")
ax.legend()
plt.tight_layout()
plt.show()

**Réponse :**

*(Les valeurs exactes sont dans le tableau ci-dessus. Les commentaires portent sur les tendances, qui ne dépendent pas du tirage.)*

- **`uniform.txt`** ($H = 8$) : aucune compression n'est possible. Chaque caractère porte déjà 8 bits d'information. Les compresseurs donnent même un facteur légèrement **inférieur à 1** : le fichier « compressé » est plus gros, à cause de l'en-tête et de la structure du format. Aucun algorithme sans perte ne peut compresser en moyenne une suite vraiment aléatoire.
- **`half.txt`** ($H = 7$) : le gain théorique est faible ($8/7 \approx 1{,}14$) et vient uniquement du bit inutilisé par caractère. Les compresseurs s'en approchent, zip presque parfaitement, mais sans jamais le dépasser.
- **`abcd.txt`** ($H = 2$) et **`abcd2.txt`** ($H = 1{,}75$) : les compresseurs atteignent une bonne partie des facteurs théoriques (4 et 4,57), mais restent **en dessous**. Ils sont aussi plus efficaces sur `abcd2` que sur `abcd` : ils exploitent bien les probabilités inégales.
- **Code OUI/NON** : appliqué directement à `abcd2.txt`, il donne environ 1,75 bit/caractère, donc un facteur proche de $8/1{,}75 \approx 4{,}57$. Il fait **mieux que zip, bz2 et xz**, car il est construit avec la connaissance exacte des $p_i$.

**Pourquoi les compresseurs ne font pas mieux.** Aucun compresseur sans perte ne peut descendre en moyenne sous $H$ bits/caractère : c'est le théorème de Shannon. Ceux-ci restent au-dessus pour trois raisons :
1. Ils ne connaissent pas la distribution à l'avance et doivent l'apprendre et la transmettre (table de Huffman, modèle adaptatif), ce qui coûte cher sur un petit fichier.
2. Ils sont conçus pour des données **corrélées** (répétitions, motifs). zip (LZ77 + Huffman) cherche des répétitions, qui n'existent pas dans une suite i.i.d.
3. Leurs codes utilisent un nombre entier de bits par symbole, et chaque format a un surcoût fixe (en-têtes, blocs).

Sur une suite de lettres indépendantes, l'entropie est donc la bonne mesure de la compressibilité, et le codage entropique adapté aux $p_i$ est optimal.

## Part IV — L'entropie de l'anglais

### \* Q1 — Rapport de compression de l'anglais
Fréquences des lettres : <http://en.wikipedia.org/wiki/Letter_frequencies>.
Calculer le rapport de compression de l'anglais (ignorer espaces, majuscules et caractères spéciaux).

In [ ]:
# Fréquences relatives des lettres en anglais (%), Wikipedia "Letter frequency"
freq_anglais = {
    "a": 8.2,   "b": 1.5,   "c": 2.8,  "d": 4.3,  "e": 12.7, "f": 2.2,
    "g": 2.0,   "h": 6.1,   "i": 7.0,  "j": 0.15, "k": 0.77, "l": 4.0,
    "m": 2.4,   "n": 6.7,   "o": 7.5,  "p": 1.9,  "q": 0.095, "r": 6.0,
    "s": 6.3,   "t": 9.1,   "u": 2.8,  "v": 0.98, "w": 2.4,  "x": 0.15,
    "y": 2.0,   "z": 0.074,
}

lettres = list(freq_anglais.keys())
p_anglais = np.array([freq_anglais[l] for l in lettres])
p_anglais = p_anglais / p_anglais.sum()          # normalisation (somme des % ≠ 100 exactement)

H_anglais = -np.sum(p_anglais * np.log2(p_anglais))
H_max = np.log2(len(lettres))                    # 26 lettres équiprobables

print(f"Entropie des lettres (modèle sans mémoire) : H = {H_anglais:.3f} bits/lettre")
print(f"Codage fixe sur 26 lettres                 : log2(26) = {H_max:.3f} bits/lettre")
print(f"Rapport de compression vs log2(26) : {H_max / H_anglais:.3f}")
print(f"Rapport de compression vs ASCII 8 bits     : {8 / H_anglais:.3f}")

In [ ]:
import heapq

def longueurs_huffman(p):
    # Longueurs des mots de code d'un code de Huffman (code préfixe optimal à longueurs entières)
    tas = [(pi, [i]) for i, pi in enumerate(p)]
    heapq.heapify(tas)
    longueurs = np.zeros(len(p), dtype=int)
    while len(tas) > 1:
        p1, g1 = heapq.heappop(tas)
        p2, g2 = heapq.heappop(tas)
        for i in g1 + g2:
            longueurs[i] += 1          # chaque fusion ajoute une question OUI/NON
        heapq.heappush(tas, (p1 + p2, g1 + g2))
    return longueurs

n_huff = longueurs_huffman(p_anglais)
n_moy = np.sum(p_anglais * n_huff)
print(f"Code de Huffman : {n_moy:.3f} bits/lettre   (H = {H_anglais:.3f} ≤ <n> < H + 1)")

# Histogramme des fréquences, trié
ordre = np.argsort(p_anglais)[::-1]
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar([lettres[i] for i in ordre], p_anglais[ordre])
ax.axhline(1/26, color="gray", ls="--", lw=0.8, label="uniforme 1/26")
ax.set_ylabel("probabilité")
ax.set_title("Fréquence des lettres en anglais")
ax.legend()
plt.tight_layout()
plt.show()

**Réponse :**

On se limite aux 26 lettres minuscules (sans espaces, majuscules ni ponctuation) et on suppose que les lettres sont tirées **indépendamment** selon leurs fréquences (modèle sans mémoire). L'entropie par lettre est

$$
H = -\sum_{i=a}^{z} p_i \log_2 p_i \approx 4{,}18 \text{ bits/lettre}.
$$

- Un codage de longueur fixe sur 26 lettres demande $\log_2 26 \approx 4{,}70$ bits/lettre. Le rapport de compression est donc $\tau = \log_2 26 / H \approx 1{,}12$. Les fréquences inégales des lettres ne font gagner qu'environ 11 %.
- Par rapport au stockage ASCII (8 bits/caractère) : $\tau = 8/H \approx 1{,}91$.
- Un code de Huffman construit sur ces fréquences atteint en pratique environ 4,2 bits/lettre, très proche de $H$, ce qui confirme $H \le \langle n \rangle < H + 1$.

Avec ce modèle, l'anglais est donc **peu compressible**. Ce résultat ne vaut cependant que si les lettres sont indépendantes, ce qui est faux (voir Q2).

### \* Q2 — Comparaison avec les meilleurs compresseurs
Voir <http://en.wikipedia.org/wiki/Hutter_Prize>. D'où vient la différence observée ?

In [ ]:
# Hutter Prize : compression du fichier enwik9 (10^9 octets extraits de Wikipedia anglais)
# Record indiqué sur la page Wikipedia au moment de la rédaction (fx2-cmix, K. Orav & B. Knoll, 2024)
# -> vérifier la valeur à jour sur http://en.wikipedia.org/wiki/Hutter_Prize
taille_enwik9 = 1_000_000_000        # octets
taille_record = 110_793_128          # octets (archive + décompresseur)

bits_par_caractere = 8 * taille_record / taille_enwik9
tau_hutter = taille_enwik9 / taille_record

print(f"Hutter Prize : {bits_par_caractere:.3f} bits/caractère, rapport de compression {tau_hutter:.2f}")
print(f"Modèle sans mémoire (Q1) : {H_anglais:.3f} bits/lettre, rapport {8/H_anglais:.2f}")
print(f"-> le meilleur compresseur utilise ~{H_anglais/bits_par_caractere:.1f}x moins de bits par caractère")

**Réponse :**

Le record du Hutter Prize compresse $10^9$ octets de Wikipedia anglais en environ $1{,}1 \times 10^8$ octets. Cela fait environ **0,9 bit par caractère**, soit un rapport de compression d'environ 9 par rapport à l'ASCII. C'est bien meilleur que les 4,18 bits/lettre (rapport 1,9) de la Q1.

**D'où vient la différence ?** L'entropie de la Q1 est celle d'un modèle où chaque lettre est **indépendante** des précédentes. Or l'anglais est très **corrélé** :
- au niveau des lettres : après « q » vient presque toujours « u », « th » et « he » sont très fréquents, certaines suites n'existent jamais ;
- au niveau des mots : le vocabulaire est fini, et une suite de lettres doit former un mot existant ;
- au niveau de la grammaire et du sens : le contexte (syntaxe, sujet du texte, mémoire à longue portée) rend la suite du texte très prévisible.

L'entropie d'une source corrélée se définit par lettre, à la limite des longs blocs :

$$
h = \lim_{n\to\infty} \frac{1}{n} H[X_1, \dots, X_n] = \lim_{n\to\infty} H[X_n \mid X_1, \dots, X_{n-1}] \le H[X].
$$

Conditionner ne peut que diminuer l'entropie : connaître le contexte réduit l'incertitude sur la lettre suivante. Les meilleurs compresseurs (cmix, modèles de type réseaux de neurones et mélange de contextes) sont avant tout d'excellents **modèles prédictifs** de la lettre suivante. Couplés à un codage arithmétique, ils codent chaque caractère avec environ $\log_2 1/p(x_n \mid \text{contexte})$ bits. C'est l'idée du Hutter Prize : bien compresser, c'est bien comprendre le texte.

*Remarques :* (i) enwik9 contient aussi des espaces, de la ponctuation et du balisage XML, donc la comparaison n'est pas faite exactement sur le même alphabet. Ces éléments sont toutefois eux-mêmes très redondants. (ii) La taille du record inclut le décompresseur, ce qui rend la comparaison honnête.

### \* Q3 — Expérience de Shannon
Estimer (et commenter) l'entropie de l'anglais avec
<https://www.csfieldguide.org.nz/en/interactives/shannon-experiment/>.

In [ ]:
# Expérience de Shannon (csfieldguide) : pour chaque lettre du texte, noter le nombre
# de tentatives nécessaires pour la deviner (1 = trouvée du premier coup).
# >>> REMPLIR avec vos propres résultats de l'expérience <<<
essais = []   # ex. [1, 1, 3, 1, 2, 1, 7, ...]

def bornes_shannon(essais, n_symboles=27):
    # Bornes de Shannon (1951) sur l'entropie par caractère à partir de la distribution
    # q_i = fraction des caractères devinés au i-ème essai (27 symboles : 26 lettres + espace)
    essais = np.asarray(essais)
    q = np.array([np.mean(essais == i) for i in range(1, n_symboles + 2)])  # q_{28} = 0
    i = np.arange(1, n_symboles + 1)
    q_i, q_suiv = q[:-1], q[1:]
    masque = q_i > 0
    borne_sup = -np.sum(q_i[masque] * np.log2(q_i[masque]))
    borne_inf = np.sum(i * (q_i - q_suiv) * np.log2(i))
    return borne_inf, borne_sup, q_i

if len(essais) == 0:
    print("Liste 'essais' vide : faites l'expérience et reportez vos résultats ci-dessus.")
else:
    b_inf, b_sup, q = bornes_shannon(essais)
    print(f"{len(essais)} caractères devinés, {np.mean(essais):.2f} essais en moyenne")
    print(f"Entropie estimée : {b_inf:.2f} ≤ h ≤ {b_sup:.2f} bits/caractère")

    fig, ax = plt.subplots(figsize=(7, 3.5))
    k_max = int(max(essais))
    ax.bar(np.arange(1, k_max + 1), q[:k_max])
    ax.set_xlabel("nombre d'essais pour deviner la lettre")
    ax.set_ylabel("fréquence $q_i$")
    ax.set_title("Expérience de Shannon")
    plt.tight_layout()
    plt.show()

**Réponse :**

**Principe.** Un humain devine un texte lettre par lettre, et on compte le nombre d'essais nécessaires pour chaque lettre. Un locuteur anglais utilise sans effort toutes les corrélations du texte (orthographe, grammaire, sens). Il se comporte donc comme un très bon prédicteur, et la suite des nombres d'essais contient la même information que le texte. Le texte peut être reconstruit par un « jumeau » qui ferait les mêmes prédictions. Si $q_i$ est la fraction des lettres trouvées au $i$-ème essai, Shannon (1951) montre que

$$
\sum_{i} i\,(q_i - q_{i+1}) \log_2 i \ \le\ h \ \le\ -\sum_i q_i \log_2 q_i .
$$

La borne supérieure est l'entropie de la suite des nombres d'essais.

**Mes résultats.** *(à compléter avec les valeurs obtenues par la cellule ci-dessus : nombre de caractères, bornes inférieure et supérieure)*

**Commentaire.**
- La majorité des lettres sont devinées dès le 1er ou le 2e essai, surtout en fin de mot. Les essais nombreux se concentrent au **début des mots** et des phrases, là où l'incertitude est la plus grande.
- Shannon trouvait environ **0,6 à 1,3 bit/caractère** pour l'anglais. C'est très inférieur aux ~4,1 bits d'un modèle de lettres indépendantes (Q1) et aux $\log_2 27 \approx 4{,}75$ bits d'un codage uniforme. C'est cohérent avec les ~0,9 bit/caractère du Hutter Prize (Q2) : l'anglais est redondant à environ 75–80 %.
- **Limites** de l'estimation : le texte est court (erreur statistique importante sur les $q_i$), le résultat dépend du texte choisi et de la personne qui devine, et seules les bornes sont accessibles, pas la valeur exacte de $h$.

---
# Exercice 2 — Estimation d'un taux de désintégration (loi de Poisson)

$P(X = k) = e^{-\lambda^*} \dfrac{(\lambda^*)^k}{k!}$, $k \in \mathbb{N}$. On prend $\lambda^* = 3$ pour les simulations.

In [ ]:
lambda_star = 3.0

### Q1 — Fonction génératrice des moments
a) Écrire $M_X(t) = \mathbb{E}[e^{tX}]$ comme une somme sur $k$.
b) Reconnaître une série usuelle et en déduire $M_X(t) = \exp\big(\lambda^*(e^t - 1)\big)$.
c) En déduire $\kappa_X(t) = \log M_X(t)$, puis $\mathbb{E}[X] = \kappa_X'(0)$ et $\mathrm{Var}(X) = \kappa_X''(0)$. Remarque ?

**Réponse :**

$$M_X(t)=\mathbb{E}[e^{tX}]=\sum_{k=0}^\infty e^{tk}e^{-\lambda^*} \frac{(\lambda^*)^k}{k!}=e^{-\lambda^*}\sum_{k=0}^\infty \frac{(\lambda^*e^t)^k}{k!}$$
On reconnait la série $\sum_{k=0}^\infty\frac{a^k}{k!}=e^a$ avec $a=\lambda^*e^t$ donc $$M_X(t)=e^{-\lambda^*}e^{\lambda^*e^t}=e^{\lambda^*(e^t-1)} \quad \text{et} \quad \kappa_X(t)=\log M_X(t)=\log e^{\lambda^*(e^t-1)}=\lambda^*(e^t-1).$$  
$\mathbb{E}[X]=\sum_{k=0}^\infty ke^{-\lambda^*}\frac{(\lambda^*)^k}{k!}=e^{-\lambda^*}\sum_{k=0}^\infty k\frac{(\lambda^*)^k}{k!}=e^{-\lambda^*}\sum_{k=1}^\infty \frac{(\lambda^*)^k}{(k-1)!}=e^{-\lambda^*}\sum_{k=0}^\infty \lambda^*\frac{(\lambda^*)^k}{k!}=\lambda^*$  
  
$\kappa'_X(t)=\lambda^*e^t \Rightarrow \kappa'_X(0)=\lambda^*=\mathbb{E}[X]$



### \* Q2 — Génération de données
Comment générer des variables de Poisson($\lambda$) ? Écrire une routine générant $N$ échantillons
(numpy, ou méthode de Knuth).

In [2]:
def poisson(lam, N):
    return np.random.poisson(lam, N)


**Réponse :**

*(à compléter)*

### Q3 — Estimateur $\hat\lambda = \frac{1}{N}\sum_i X_i$
Moyenne ? Variance ? Erreur typique ?

**Réponse :**

*(à compléter)*

### \* Q4 — Convergence numérique
Vérifier que $\hat\lambda \to \lambda^*$ quand $N$ augmente. Tracer la distribution empirique de $\hat\lambda$
pour différentes valeurs de $N$ (répéter l'expérience $M$ fois à $N$ fixé).

In [ ]:
N_valeurs = [10, 100, 1000]   # à ajuster
M = 10_000                    # nombre de répétitions à N fixé (à ajuster)

def estimations_lambda(lam, N, M):
    # TODO : renvoyer un tableau de M valeurs de lambda_hat
    pass

In [ ]:
# TODO : graphiques (histogrammes de lambda_hat pour chaque N, écart à lambda* en fonction de N, ...)

**Réponse :**

*(à compléter)*

### \* Q5 — Théorème central limite
Rappeler le TCL dans ce cadre et préciser la loi limite de $S_N = \sqrt{N}\,\hat\lambda_N$.
Qu'est-ce que cela implique pour la loi de $\hat\lambda_N$ ? Vérifier numériquement.

**Réponse :**

*(à compléter)*

In [ ]:
# TODO : histogrammes numériques comparés à la loi limite prédite

### Q6 — Grandes déviations
$P(\hat\lambda_N \approx s) \asymp e^{-N I(s)}$, avec (Cramér) $I(s) = \sup_{t \in \mathbb{R}} \{ ts - \kappa_X(t) \}$.
Calculer $I(s)$.

**Réponse :**

*(à compléter)*

### Q7 — Développement autour de la moyenne
Développer $I(s)$ au voisinage de $s = \lambda^*$. Montrer qu'on retrouve l'approximation gaussienne du TCL.
Commenter les déviations par rapport à la gaussienne.

**Réponse :**

*(à compléter)*

### \* Q8 — Vérification numérique
Pour plusieurs $N$, simuler un grand nombre d'expériences : comparer l'histogramme de $S_N$ à la gaussienne du TCL.
Jusqu'où l'approximation gaussienne est-elle valable ? Vérifier la loi des grandes déviations.

In [ ]:
def I_cramer(s, lam):
    # TODO : fonction de taux obtenue en Q6
    pass

def I_gauss(s, lam):
    # TODO : approximation quadratique obtenue en Q7
    pass

In [ ]:
# TODO : simulations pour plusieurs N (grand nombre d'expériences)

In [ ]:
# TODO : histogramme de S_N vs gaussienne du TCL (échelle log conseillée pour voir les queues)

In [ ]:
# TODO : comparer -(1/N) log P(lambda_hat ≈ s) empirique à I(s) et à l'approximation gaussienne

**Réponse :**

*(à compléter)*

---
## Conclusion
*(à compléter)*